# Electricity Foundation Models — Zero-Shot Forecasting

**Role**  
Evaluate whether general-purpose pretrained time-series foundation models can forecast South Australian electricity demand competitively without Electricity-specific fitting.

**Models**  
Chronos-Bolt-Tiny and TimesFM.

**Evaluation**  
Protocol A — rolling one-step. Protocol B — 48-step day-ahead.

**Inputs**  
`data/electricity/australian_electricity_demand_dataset.tsf`; frozen Protocol A and Protocol B baseline, DHR-ARIMA, LSTM, Chronos, and TimesFM forecast CSVs; `results/electricity/uncertainty_summary.csv`; and the two saved Protocol A quantile checkpoints.

**Outputs**  
This notebook reads and validates the frozen artifacts above. Optional regeneration, disabled by default, writes candidates only beneath `results/electricity/staging/foundation_models/`; it never promotes them automatically.

**Depends On**  
`10_Electricity_EDA.ipynb` · `11_Electricity_Classical_Baselines.ipynb` · `12_Electricity_LSTM.ipynb`

**Authoritative Status**  
The normal execution path reads frozen authoritative forecast candidates and the downstream uncertainty summary. Regeneration capability is retained only as an explicit, staged, non-authoritative path.

**What This Notebook Does Not Do**

- fit Chronos or TimesFM to Electricity;
- calculate final Trust Scores;
- perform final robustness analysis or statistical significance testing;
- claim universal foundation-model superiority.


## 1. Objective

**Research question.** Can general-purpose zero-shot time-series foundation models produce competitive South Australian electricity-demand forecasts under both rolling one-step and genuine 24-hour day-ahead information constraints, without Electricity-specific parameter training?

Secondary questions:

1. Does performance change materially between Protocol A and Protocol B?
2. How do Chronos and TimesFM compare with strong protocol-appropriate baselines?
3. Does better point accuracy correspond to better probabilistic calibration?
4. What reproducibility risks arise from external pretrained checkpoints?


## 2. Setup


In [ ]:
from pathlib import Path
import hashlib, importlib.metadata as md, platform, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "src").is_dir() and (candidate / "results").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root.")

ROOT = find_project_root(Path.cwd())
DATA = ROOT / "data/electricity/australian_electricity_demand_dataset.tsf"
RESULTS = ROOT / "results/electricity"
CONTEXT, HORIZON = 336, 48
MODEL_COLUMNS = {"Chronos-Bolt-Tiny": "Chronos_Bolt_Tiny", "TimesFM": "TimesFM"}
MODEL_COLORS = {"Chronos-Bolt-Tiny": "#0072B2", "TimesFM": "#D55E00"}
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (10, 4), "axes.titlesize": 12, "axes.labelsize": 10})

def package_version(name):
    try: return md.version(name)
    except md.PackageNotFoundError: return "Not installed in review environment"

environment = pd.DataFrame({"Item": ["Python", "Platform", "chronos-forecasting", "timesfm"],
                            "Value": [platform.python_version(), platform.platform(), package_version("chronos-forecasting"), package_version("timesfm")]})
display(environment)


In [ ]:
def load_tsf(path):
    attributes, rows = [], []
    with Path(path).open(encoding="utf-8") as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if not line or line.startswith("#"): continue
            if line.startswith("@attribute"):
                _, name, kind = line.split(maxsplit=2); attributes.append((name, kind))
            elif not line.startswith("@"):
                parts = line.split(":", len(attributes))
                record = dict(zip((a[0] for a in attributes), parts[:-1]))
                record["series_value"] = np.fromstring(parts[-1], sep=",")
                rows.append(record)
    return pd.DataFrame(rows)

def metric_row(actual, forecast, scale):
    a, p = np.asarray(actual, float), np.asarray(forecast, float)
    error = a - p
    return {"MAE": np.mean(np.abs(error)), "RMSE": np.sqrt(np.mean(error ** 2)),
            "MAPE": np.mean(np.abs(error / a)) * 100,
            "sMAPE": np.mean(2 * np.abs(error) / (np.abs(a) + np.abs(p))) * 100,
            "MASE-48": np.mean(np.abs(error)) / scale}

def residual_row(actual, forecast):
    error = np.asarray(actual, float) - np.asarray(forecast, float)
    return {"Mean error": error.mean(), "Median error": np.median(error),
            "Residual standard deviation": error.std(ddof=1),
            "MAE": np.mean(np.abs(error)), "RMSE": np.sqrt(np.mean(error ** 2))}

def sha256(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def show_table(frame, digits=4): display(frame.round(digits))


## 3. Data and Forecasting Protocol

### 3.1 Dataset and Frozen Evaluation Partitions

The South Australian `T4` half-hourly series is partitioned chronologically. Development and validation precede the final pre-test boundary; no model fitting or tuning is performed on the final test set.


In [ ]:
raw = load_tsf(DATA)
selected = raw[(raw.series_name == "T4") & (raw.state == "SA")]
assert len(selected) == 1
row = selected.iloc[0]
index = pd.date_range(pd.to_datetime(row.start_timestamp, format="%Y-%m-%d %H-%M-%S"), periods=len(row.series_value), freq="30min")
y = pd.Series(row.series_value, index=index, name="Actual")
dev = y.loc[:"2011-06-23 23:30"]
validation = y.loc["2011-06-24":"2012-07-12 23:30"]
pretest = y.loc[:"2012-07-12 23:30"]
test = y.loc["2012-07-13":"2015-03-01 23:30"]
partitions = [("Development", dev, "Model development / selection"), ("Validation", validation, "Selection without final-test access"),
              ("Pre-test", pretest, "All observations legally available before test"), ("Final test", test, "Frozen out-of-sample evaluation")]
partition_table = pd.DataFrame([{"Partition": name, "Start": s.index.min(), "End": s.index.max(), "Number of observations": len(s), "Purpose": purpose} for name, s, purpose in partitions])
assert tuple(partition_table["Number of observations"]) == (166128, 18480, 184608, 46176)
scale48 = float(np.mean(np.abs(pretest.to_numpy()[48:] - pretest.to_numpy()[:-48])))
show_table(partition_table, 3)
display(pd.DataFrame({"Definition": ["mean(|y[t] - y[t-48]|) on pre-test data"], "MASE-48 denominator": [scale48]}))


### 3.2 Protocol A — Rolling One-Step

At each half-hour, the model forecasts target `t`; only after that forecast is fixed is actual `t` revealed, and it may then enter the context for `t+1`.


In [ ]:
protocol_a_summary = pd.DataFrame({"Characteristic": ["Forecast horizon", "Update frequency", "Within-horizon actual updates", "Operational interpretation", "Number of forecasts"],
    "Protocol A": ["1 half-hour step", "Every 30 minutes", "Yes—after each forecast", "Rolling short-horizon operation", len(test)]})
display(protocol_a_summary)


### 3.3 Protocol B — 48-Step Day-Ahead

At each midnight origin, the model generates all 48 half-hour predictions from the same legally available context. No actual observation inside that 24-hour horizon may update the context.


In [ ]:
origins = test.index[::HORIZON]
protocol_b_summary = pd.DataFrame({"Characteristic": ["Forecast horizon", "Update frequency", "Within-horizon actual updates", "Operational interpretation", "Number of forecasts"],
    "Protocol B": ["48 half-hour steps (24 hours)", "Once per day at midnight", "No", "Genuine day-ahead operation", f"{len(origins)} origins × 48 = {len(test)} values"]})
assert len(origins) == 962
display(protocol_b_summary)


### 3.4 Why Both Protocols Are Necessary

Frequent updating can make a model appear strong because new actuals continually correct its context. Day-ahead forecasting removes that advantage and exposes error accumulation across the full operational horizon. Keeping both protocols separate is therefore central to trustworthy benchmarking.


In [ ]:
display(pd.DataFrame({"Feature": ["Horizon", "Forecast origins", "Actual updates", "Operational difficulty", "Primary interpretation"],
    "Protocol A": ["1 step", len(test), "After every forecast", "Lower; context refreshes", "Rolling 30-minute accuracy"],
    "Protocol B": ["48 steps", len(origins), "None within 24 hours", "Higher; errors may accumulate", "True day-ahead accuracy"]}))


## 4. Foundation-Model Configuration

### 4.1 Zero-Shot Evaluation Principle

“Zero-shot” means the pretrained checkpoint parameters are used as released: Electricity observations form the input context, but no Electricity-specific gradient update, fine-tuning, or parameter selection is performed.

### 4.2 Model Configuration — Table


In [ ]:
configuration = pd.DataFrame([
 {"Model":"Chronos-Bolt-Tiny", "Checkpoint":"amazon/chronos-bolt-tiny", "Revision":"Not pinned / not recorded", "Model family":"Chronos Bolt", "Zero-shot":True, "Context length":CONTEXT, "Protocol A horizon":1, "Protocol B horizon":48, "Device":"CPU", "Numeric precision":"torch.float32", "Native probabilistic output":"Quantiles, including 0.1/0.5/0.9", "Electricity-specific training":"None"},
 {"Model":"TimesFM", "Checkpoint":"google/timesfm-2.5-200m-pytorch", "Revision":"Not pinned / not recorded", "Model family":"TimesFM 2.5 (200M, PyTorch)", "Zero-shot":True, "Context length":CONTEXT, "Protocol A horizon":1, "Protocol B horizon":48, "Device":"CPU / framework default", "Numeric precision":"Not explicitly recorded", "Native probabilistic output":"Point plus quantiles, including 0.1/0.9", "Electricity-specific training":"None"}])
display(configuration)


### 4.3 Information-Set Guardrail

Both models receive only observations legally available at the forecast origin. Protocol A contexts may incorporate the previous revealed actual; Protocol B contexts remain fixed for all 48 predictions. This is the same leakage discipline applied to the classical and LSTM comparisons.


In [ ]:
artifact_paths = {
 "A baseline": RESULTS/"protocol_a_baseline_forecasts.csv", "A DHR": RESULTS/"protocol_a_dhr_forecast.csv", "A LSTM": RESULTS/"protocol_a_lstm_forecast.csv",
 "A Chronos": RESULTS/"protocol_a_chronos_forecast.csv", "A TimesFM": RESULTS/"protocol_a_timesfm_forecast.csv",
 "B baseline": RESULTS/"protocol_b_baseline_forecasts.csv", "B DHR": RESULTS/"protocol_b_dhr_forecast.csv", "B LSTM": RESULTS/"protocol_b_lstm_forecast.csv",
 "B Chronos": RESULTS/"protocol_b_chronos_forecast.csv", "B TimesFM": RESULTS/"protocol_b_timesfm_forecast.csv",
 "Uncertainty summary": RESULTS/"uncertainty_summary.csv", "A Chronos intervals": RESULTS/".phase5_chronos_a_checkpoint.npz", "A TimesFM intervals": RESULTS/".phase5_timesfm_a_checkpoint.npz"}
assert all(p.exists() for p in artifact_paths.values())

pa = pd.read_csv(artifact_paths["A baseline"], parse_dates=["Timestamp"])
for label, key in [("DHR_ARIMA","A DHR"), ("LSTM","A LSTM"), ("Chronos_Bolt_Tiny","A Chronos"), ("TimesFM","A TimesFM")]:
    pa = pa.merge(pd.read_csv(artifact_paths[key], parse_dates=["Timestamp"]), on="Timestamp", validate="one_to_one")
keys = ["Origin", "Timestamp", "Horizon"]
pb = pd.read_csv(artifact_paths["B baseline"], parse_dates=["Origin", "Timestamp"])
for label, key in [("DHR_ARIMA","B DHR"), ("LSTM","B LSTM"), ("Chronos_Bolt_Tiny","B Chronos"), ("TimesFM","B TimesFM")]:
    pb = pb.merge(pd.read_csv(artifact_paths[key], parse_dates=["Origin", "Timestamp"]), on=keys, validate="one_to_one")
assert pa.Timestamp.equals(pd.Series(test.index, name="Timestamp"))
assert pb.Timestamp.equals(pd.Series(test.index, name="Timestamp"))
assert np.array_equal(pa.Actual.to_numpy(), test.to_numpy()) and np.array_equal(pb.Actual.to_numpy(), test.to_numpy())
assert pb.Origin.nunique() == 962 and pb.groupby("Origin").size().eq(48).all()
assert pb.groupby("Origin").Horizon.apply(lambda s: s.tolist() == list(range(1, 49))).all()
assert np.isfinite(pa[list(MODEL_COLUMNS.values())].to_numpy()).all() and np.isfinite(pb[list(MODEL_COLUMNS.values())].to_numpy()).all()


## 5. Chronos-Bolt-Tiny — Individual Model Analysis

### 5.1 Model Method

The `amazon/chronos-bolt-tiny` checkpoint is used without fitting, with 336 legally available observations. Protocol A requests one step; Protocol B requests one 48-step vector. The point forecast is the native median (0.5 quantile); native 0.1 and 0.9 quantiles define the available 80% interval evidence.


In [ ]:
chronos_bolt_tiny_a_metrics = pd.DataFrame([{"Model": "Chronos-Bolt-Tiny", **metric_row(pa.Actual, pa["Chronos_Bolt_Tiny"], scale48)}])


### 5.2 Protocol A Results — Table

All metrics are recomputed from the frozen Protocol A forecast artifact; MASE-48 uses the unchanged pre-test seasonal-naive denominator.


In [ ]:
show_table(chronos_bolt_tiny_a_metrics)


### 5.3 Protocol A Forecast — Visualization

The first complete seven-day test interval is shown by a fixed, non-performance-based rule to keep the dense series readable.


In [ ]:
segment = pa.iloc[:7*48]
ax = segment.plot(x="Timestamp", y=["Actual", "Chronos_Bolt_Tiny"], color=["black", MODEL_COLORS["Chronos-Bolt-Tiny"]], linewidth=1.0)
ax.set(title="Chronos-Bolt-Tiny: Protocol A — first seven test days", xlabel="Timestamp", ylabel="Demand (MW)")
ax.legend(["Actual", "Chronos-Bolt-Tiny"]); plt.tight_layout(); plt.show()


### 5.4 Protocol A Residual Diagnostics

Residuals are defined as actual minus forecast. This is basic point-forecast evidence, not the downstream robustness analysis.


In [ ]:
residual = pa.Actual - pa["Chronos_Bolt_Tiny"]
ax = pa.assign(Residual=residual).plot(x="Timestamp", y="Residual", color=MODEL_COLORS["Chronos-Bolt-Tiny"], linewidth=.4, legend=False)
ax.axhline(0, color="black", linewidth=.8); ax.set(title="Chronos-Bolt-Tiny: Protocol A residuals", xlabel="Timestamp", ylabel="Actual − forecast (MW)")
plt.tight_layout(); plt.show()
show_table(pd.DataFrame([{"Model":"Chronos-Bolt-Tiny", **residual_row(pa.Actual, pa["Chronos_Bolt_Tiny"])}]))


### 5.5 Protocol B Results — Table


In [ ]:
chronos_bolt_tiny_b_metrics = pd.DataFrame([{"Model": "Chronos-Bolt-Tiny", **metric_row(pb.Actual, pb["Chronos_Bolt_Tiny"], scale48)}])
show_table(chronos_bolt_tiny_b_metrics)


### 5.6 Protocol B Day-Ahead Example — Visualization

The representative origin is selected once for both models as the day whose **mean two-model absolute error** is closest to the median across all origins. This deterministic joint rule avoids choosing an unusually favorable day for either model.


In [ ]:
if "representative_origin" not in globals():
    day_error = pb.assign(joint_abs_error=((pb.Actual-pb.Chronos_Bolt_Tiny).abs() + (pb.Actual-pb.TimesFM).abs())/2).groupby("Origin").joint_abs_error.mean()
    representative_origin = (day_error - day_error.median()).abs().idxmin()
example = pb[pb.Origin == representative_origin]
ax = example.plot(x="Horizon", y=["Actual", "Chronos_Bolt_Tiny"], marker="o", color=["black", MODEL_COLORS["Chronos-Bolt-Tiny"]])
ax.set(title=f"Chronos-Bolt-Tiny: representative Protocol B origin {representative_origin:%Y-%m-%d}", xlabel="Half-hour horizon", ylabel="Demand (MW)")
ax.legend(["Actual", "Chronos-Bolt-Tiny"]); plt.tight_layout(); plt.show()


### 5.7 Horizon-Wise Error — Visualization

MAE is recomputed separately at each of the 48 horizons from the frozen forecasts.


In [ ]:
chronos_bolt_tiny_horizon = pd.DataFrame([{"Horizon": h, "MAE": np.mean(np.abs(g.Actual-g["Chronos_Bolt_Tiny"]))} for h,g in pb.groupby("Horizon")])
ax = chronos_bolt_tiny_horizon.plot(x="Horizon", y="MAE", color=MODEL_COLORS["Chronos-Bolt-Tiny"], legend=False)
ax.set(title="Chronos-Bolt-Tiny: Protocol B MAE by horizon", xlabel="Half-hour horizon", ylabel="MAE (MW)"); plt.tight_layout(); plt.show()


## 6. TimesFM — Individual Model Analysis

### 6.1 Model Method

The `google/timesfm-2.5-200m-pytorch` checkpoint is compiled for maximum context 336 and maximum horizon 48, with input normalization enabled and no fitting. Protocol A requests one step; Protocol B requests one 48-step vector. The API point output is retained, while native 0.1 and 0.9 quantiles define the available 80% interval evidence.


In [ ]:
timesfm_a_metrics = pd.DataFrame([{"Model": "TimesFM", **metric_row(pa.Actual, pa["TimesFM"], scale48)}])


### 6.2 Protocol A Results — Table

All metrics are recomputed from the frozen Protocol A forecast artifact; MASE-48 uses the unchanged pre-test seasonal-naive denominator.


In [ ]:
show_table(timesfm_a_metrics)


### 6.3 Protocol A Forecast — Visualization

The first complete seven-day test interval is shown by a fixed, non-performance-based rule to keep the dense series readable.


In [ ]:
segment = pa.iloc[:7*48]
ax = segment.plot(x="Timestamp", y=["Actual", "TimesFM"], color=["black", MODEL_COLORS["TimesFM"]], linewidth=1.0)
ax.set(title="TimesFM: Protocol A — first seven test days", xlabel="Timestamp", ylabel="Demand (MW)")
ax.legend(["Actual", "TimesFM"]); plt.tight_layout(); plt.show()


### 6.4 Protocol A Residual Diagnostics

Residuals are defined as actual minus forecast. This is basic point-forecast evidence, not the downstream robustness analysis.


In [ ]:
residual = pa.Actual - pa["TimesFM"]
ax = pa.assign(Residual=residual).plot(x="Timestamp", y="Residual", color=MODEL_COLORS["TimesFM"], linewidth=.4, legend=False)
ax.axhline(0, color="black", linewidth=.8); ax.set(title="TimesFM: Protocol A residuals", xlabel="Timestamp", ylabel="Actual − forecast (MW)")
plt.tight_layout(); plt.show()
show_table(pd.DataFrame([{"Model":"TimesFM", **residual_row(pa.Actual, pa["TimesFM"])}]))


### 6.5 Protocol B Results — Table


In [ ]:
timesfm_b_metrics = pd.DataFrame([{"Model": "TimesFM", **metric_row(pb.Actual, pb["TimesFM"], scale48)}])
show_table(timesfm_b_metrics)


### 6.6 Protocol B Day-Ahead Example — Visualization

The representative origin is selected once for both models as the day whose **mean two-model absolute error** is closest to the median across all origins. This deterministic joint rule avoids choosing an unusually favorable day for either model.


In [ ]:
if "representative_origin" not in globals():
    day_error = pb.assign(joint_abs_error=((pb.Actual-pb.Chronos_Bolt_Tiny).abs() + (pb.Actual-pb.TimesFM).abs())/2).groupby("Origin").joint_abs_error.mean()
    representative_origin = (day_error - day_error.median()).abs().idxmin()
example = pb[pb.Origin == representative_origin]
ax = example.plot(x="Horizon", y=["Actual", "TimesFM"], marker="o", color=["black", MODEL_COLORS["TimesFM"]])
ax.set(title=f"TimesFM: representative Protocol B origin {representative_origin:%Y-%m-%d}", xlabel="Half-hour horizon", ylabel="Demand (MW)")
ax.legend(["Actual", "TimesFM"]); plt.tight_layout(); plt.show()


### 6.7 Horizon-Wise Error — Visualization

MAE is recomputed separately at each of the 48 horizons from the frozen forecasts.


In [ ]:
timesfm_horizon = pd.DataFrame([{"Horizon": h, "MAE": np.mean(np.abs(g.Actual-g["TimesFM"]))} for h,g in pb.groupby("Horizon")])
ax = timesfm_horizon.plot(x="Horizon", y="MAE", color=MODEL_COLORS["TimesFM"], legend=False)
ax.set(title="TimesFM: Protocol B MAE by horizon", xlabel="Half-hour horizon", ylabel="MAE (MW)"); plt.tight_layout(); plt.show()


## 7. Foundation-Model Comparative Analysis

### 7.1 Protocol A Comparison — Table


In [ ]:
fm_a = pd.DataFrame([{"Model": m, **metric_row(pa.Actual, pa[c], scale48)} for m,c in MODEL_COLUMNS.items()]).sort_values("MASE-48").reset_index(drop=True)
fm_a.insert(0, "Rank within foundation models", np.arange(1, len(fm_a)+1)); show_table(fm_a)


### 7.2 Protocol A Comparison — Visualization


In [ ]:
ax = fm_a.sort_values("MASE-48", ascending=False).plot.barh(x="Model", y="MASE-48", color=[MODEL_COLORS[m] for m in fm_a.sort_values("MASE-48", ascending=False).Model], legend=False)
ax.set(title="Protocol A foundation-model comparison", xlabel="MASE-48 (lower is better)", ylabel=""); plt.tight_layout(); from pathlib import Path
_figure_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "figures").is_dir())
_figure_dir = _figure_root / "figures" / "electricity"
_figure_dir.mkdir(parents=True, exist_ok=True)
_figure_path = _figure_dir / "electricity_foundation_protocol_a_comparison.png"
plt.gcf().savefig(_figure_path, dpi=300, bbox_inches="tight")
plt.show()


### 7.3 Protocol B Comparison — Table


In [ ]:
fm_b = pd.DataFrame([{"Model": m, **metric_row(pb.Actual, pb[c], scale48)} for m,c in MODEL_COLUMNS.items()]).sort_values("MASE-48").reset_index(drop=True)
fm_b.insert(0, "Rank within foundation models", np.arange(1, len(fm_b)+1)); show_table(fm_b)


### 7.4 Protocol B Comparison — Visualization


In [ ]:
ax = fm_b.sort_values("MASE-48", ascending=False).plot.barh(x="Model", y="MASE-48", color=[MODEL_COLORS[m] for m in fm_b.sort_values("MASE-48", ascending=False).Model], legend=False)
ax.set(title="Protocol B foundation-model comparison", xlabel="MASE-48 (lower is better)", ylabel=""); plt.tight_layout(); from pathlib import Path
_figure_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "figures").is_dir())
_figure_dir = _figure_root / "figures" / "electricity"
_figure_dir.mkdir(parents=True, exist_ok=True)
_figure_path = _figure_dir / "electricity_foundation_protocol_b_comparison.png"
plt.gcf().savefig(_figure_path, dpi=300, bbox_inches="tight")
plt.show()


### 7.5 Horizon-Wise Head-to-Head

This direct comparison tests whether the relative advantage is stable or changes across the day-ahead horizon.


In [ ]:
horizon_head = pd.concat([chronos_bolt_tiny_horizon.assign(Model="Chronos-Bolt-Tiny"), timesfm_horizon.assign(Model="TimesFM")])
fig, ax = plt.subplots()
for model, g in horizon_head.groupby("Model"):
    ax.plot(g.Horizon, g.MAE, label=model, color=MODEL_COLORS[model])
ax.set(title="Protocol B horizon-wise head-to-head", xlabel="Half-hour horizon", ylabel="MAE (MW)"); ax.legend(); plt.tight_layout(); from pathlib import Path
_figure_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "figures").is_dir())
_figure_dir = _figure_root / "figures" / "electricity"
_figure_dir.mkdir(parents=True, exist_ok=True)
_figure_path = _figure_dir / "electricity_foundation_horizon_comparison.png"
plt.gcf().savefig(_figure_path, dpi=300, bbox_inches="tight")
plt.show()
hwin = horizon_head.pivot(index="Horizon", columns="Model", values="MAE")
display(Markdown(f"**Evidence.** Chronos has lower horizon-specific MAE at **{int((hwin['Chronos-Bolt-Tiny'] < hwin['TimesFM']).sum())} of 48** horizons; TimesFM has lower MAE at **{int((hwin['TimesFM'] < hwin['Chronos-Bolt-Tiny']).sum())} of 48** horizons."))


## 8. Context Against Existing Baselines

The tables deliberately include only the strongest protocol-appropriate classical comparator, the LSTM, and the two foundation models. Each protocol is ranked within its own information set.

### 8.1 Protocol A Context — Table


In [ ]:
baseline_cols = {"Naive":"Naive", "Daily Seasonal Naive":"Daily_Seasonal_Naive", "Weekly Seasonal Naive":"Weekly_Seasonal_Naive", "Moving Average":"Moving_Average", "DHR-ARIMA":"DHR_ARIMA"}
baseline_a_all = pd.DataFrame([{"Model":m, **metric_row(pa.Actual, pa[c], scale48)} for m,c in baseline_cols.items()]).sort_values("MASE-48")
best_a_baseline = baseline_a_all.iloc[0].Model
context_a_models = {best_a_baseline: baseline_cols[best_a_baseline], "LSTM":"LSTM", **MODEL_COLUMNS}
context_a = pd.DataFrame([{"Model":m, **metric_row(pa.Actual, pa[c], scale48)} for m,c in context_a_models.items()]).sort_values("MASE-48")
show_table(context_a)


### 8.2 Protocol B Context — Table


In [ ]:
baseline_b_all = pd.DataFrame([{"Model":m, **metric_row(pb.Actual, pb[c], scale48)} for m,c in baseline_cols.items()]).sort_values("MASE-48")
best_b_baseline = baseline_b_all.iloc[0].Model
context_b_models = {best_b_baseline: baseline_cols[best_b_baseline], "LSTM":"LSTM", **MODEL_COLUMNS}
context_b = pd.DataFrame([{"Model":m, **metric_row(pb.Actual, pb[c], scale48)} for m,c in context_b_models.items()]).sort_values("MASE-48")
show_table(context_b)


### 8.3 Relative Improvement Over Protocol-Specific Baseline

Relative improvement is `(baseline MASE-48 − model MASE-48) / baseline MASE-48 × 100`; negative values indicate deterioration.


In [ ]:
relative_rows=[]
for protocol, frame, actual, best, all_base in [("A", pa, pa.Actual, best_a_baseline, baseline_a_all), ("B", pb, pb.Actual, best_b_baseline, baseline_b_all)]:
    baseline_mase = float(all_base.set_index("Model").loc[best, "MASE-48"])
    for model,col in MODEL_COLUMNS.items():
        model_mase = metric_row(actual, frame[col], scale48)["MASE-48"]
        relative_rows.append({"Model":model, "Protocol":protocol, "Reference baseline":best, "Model MASE-48":model_mase, "Baseline MASE-48":baseline_mase, "Relative improvement %":(baseline_mase-model_mase)/baseline_mase*100})
relative_improvement = pd.DataFrame(relative_rows); show_table(relative_improvement)


## 9. Point Accuracy vs Probabilistic Evidence

### 9.1 Available Native Uncertainty

Both foundation models expose native quantile evidence. This notebook reports only the preserved 0.1/0.9 (nominal 80%) aggregate evidence; it constructs no intervals for deterministic models and invents no 95% intervals. Protocol A quantile vectors are preserved in checkpoint NPZ files. Protocol B interval vectors were not preserved, so only the authoritative aggregate summary is available here.

### 9.2 Native 80% Interval Summary — Table


In [ ]:
uncertainty_raw = pd.read_csv(RESULTS/"uncertainty_summary.csv")
uncertainty = uncertainty_raw[(uncertainty_raw.Model.isin(MODEL_COLUMNS.values())) & (uncertainty_raw.Interval == "80%") & uncertainty_raw.Available].copy()
uncertainty["Model"] = uncertainty.Model.map({v:k for k,v in MODEL_COLUMNS.items()})
uncertainty["Coverage error"] = uncertainty.Empirical_Coverage - uncertainty.Nominal_Coverage
uncertainty_summary = uncertainty.rename(columns={"Nominal_Coverage":"Nominal coverage", "Empirical_Coverage":"Empirical coverage", "Average_Width":"Average interval width"})
show_table(uncertainty_summary[["Model","Protocol","Nominal coverage","Empirical coverage","Coverage error","Average interval width"]])


### 9.3 Accuracy–Calibration Trade-Off

Better point accuracy does not necessarily imply better calibration: the criteria measure different properties and should be inspected together, not collapsed into a Trust Score here.


In [ ]:
accuracy = pd.concat([fm_a.assign(Protocol="A"), fm_b.assign(Protocol="B")])[["Model","Protocol","MASE-48"]]
tradeoff = accuracy.merge(uncertainty_summary[["Model","Protocol","Empirical coverage","Average interval width"]], on=["Model","Protocol"])
tradeoff["Absolute coverage error"] = (tradeoff["Empirical coverage"] - .8).abs()
show_table(tradeoff[["Model","Protocol","MASE-48","Empirical coverage","Absolute coverage error","Average interval width"]])


## 10. Diagnostic Comparison

### 10.1 Residual Comparison — Visualization


In [ ]:
fig, axes = plt.subplots(2,1,figsize=(11,7),sharex=False)
for ax, frame, protocol in [(axes[0],pa,"A"),(axes[1],pb,"B")]:
    for model,col in MODEL_COLUMNS.items(): ax.plot(frame.Timestamp, frame.Actual-frame[col], label=model, color=MODEL_COLORS[model], linewidth=.35, alpha=.8)
    ax.axhline(0,color="black",linewidth=.7); ax.set(title=f"Protocol {protocol} residual comparison", ylabel="Actual − forecast (MW)"); ax.legend()
axes[-1].set_xlabel("Timestamp"); plt.tight_layout(); from pathlib import Path
_figure_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "figures").is_dir())
_figure_dir = _figure_root / "figures" / "electricity"
_figure_dir.mkdir(parents=True, exist_ok=True)
_figure_path = _figure_dir / "electricity_foundation_residuals.png"
plt.gcf().savefig(_figure_path, dpi=300, bbox_inches="tight")
plt.show()


### 10.2 Error Distribution — Visualization

Empirical CDFs show whether differences are broad-based or concentrated in the largest misses.


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(11,4))
for ax, frame, protocol in [(axes[0],pa,"A"),(axes[1],pb,"B")]:
    for model,col in MODEL_COLUMNS.items():
        values=np.sort(np.abs(frame.Actual-frame[col])); ax.plot(values,np.arange(1,len(values)+1)/len(values),label=model,color=MODEL_COLORS[model])
    ax.set(title=f"Protocol {protocol} absolute-error ECDF", xlabel="Absolute error (MW)", ylabel="Cumulative proportion"); ax.legend()
plt.tight_layout(); plt.show()


### 10.3 Horizon Degradation — Table


In [ ]:
blocks = pd.cut(pb.Horizon, [0,12,24,36,48], labels=["1–12","13–24","25–36","37–48"])
rows=[]
for model,col in MODEL_COLUMNS.items():
    for block,g in pb.groupby(blocks, observed=True): rows.append({"Model":model,"Horizons":str(block),"MAE":np.mean(np.abs(g.Actual-g[col])),"MASE-48":np.mean(np.abs(g.Actual-g[col]))/scale48})
horizon_blocks=pd.DataFrame(rows); show_table(horizon_blocks)


## 11. Reproducibility and Artifact Provenance

### 11.1 Reproducibility Record — Table


In [ ]:
reproducibility = pd.DataFrame([
 {"Model":"Chronos-Bolt-Tiny","Checkpoint":"amazon/chronos-bolt-tiny","Checkpoint revision pinned?":"No / not recorded","Library/package":"chronos-forecasting","Library version":package_version("chronos-forecasting"),"Device":"CPU","Precision":"float32","Context length":336,"Zero-shot":True,"Forecast artifacts preserved?":True,"Native interval artifacts preserved?":"A vectors + A/B aggregate","Expensive inference required for normal reading?":False},
 {"Model":"TimesFM","Checkpoint":"google/timesfm-2.5-200m-pytorch","Checkpoint revision pinned?":"No / not recorded","Library/package":"timesfm","Library version":package_version("timesfm"),"Device":"CPU / framework default","Precision":"Not explicitly recorded","Context length":336,"Zero-shot":True,"Forecast artifacts preserved?":True,"Native interval artifacts preserved?":"A vectors + A/B aggregate","Expensive inference required for normal reading?":False}])
display(reproducibility)


### 11.2 Frozen Artifact Inventory — Table


In [ ]:
inventory=[]
for purpose,path in artifact_paths.items():
    if path.suffix == ".csv":
        frame=pd.read_csv(path); rows=len(frame); columns=", ".join(frame.columns)
    else:
        with np.load(path) as bundle: rows=len(bundle[bundle.files[0]]); columns=", ".join(bundle.files)
    inventory.append({"Artifact":path.name,"Purpose":purpose,"Rows":rows,"Key columns / arrays":columns,"Status":"Frozen; read only","SHA-256":sha256(path)})
artifact_inventory=pd.DataFrame(inventory); display(artifact_inventory)


### 11.3 Artifact-First Execution Principle

Normal research review relies on frozen artifacts. Model packages and checkpoint downloads are not required to reproduce tables, metrics, diagnostics, plots, or downstream comparisons. Expensive inference is isolated in the final regeneration section and disabled by default.


## 12. Safe Regeneration Controls

Regeneration is retained for reproducibility but is not part of normal execution. Candidates are written only to staging. Promotion is intentionally disabled and no authoritative artifact is overwritten.

### 12.1 Candidate Generation


In [ ]:
RUN_GENERATION = False
PROMOTE_TO_AUTHORITATIVE = False
STAGING = RESULTS / "staging/foundation_models"
assert RUN_GENERATION is False and PROMOTE_TO_AUTHORITATIVE is False

if RUN_GENERATION:
    import torch
    from chronos import ChronosBoltPipeline
    import timesfm
    STAGING.mkdir(parents=True, exist_ok=True)
    values=y.to_numpy(float); positions=pd.Series(np.arange(len(y)),index=y.index)
    test_positions=positions.loc[test.index].to_numpy(); origin_positions=positions.loc[origins].to_numpy()
    contexts_a=np.stack([values[p-CONTEXT:p] for p in test_positions]).astype(np.float32)
    contexts_b=np.stack([values[p-CONTEXT:p] for p in origin_positions]).astype(np.float32)
    chronos_model=ChronosBoltPipeline.from_pretrained("amazon/chronos-bolt-tiny",device_map="cpu",dtype=torch.float32)
    timesfm_model=timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch",torch_compile=False)
    timesfm_model.compile(timesfm.ForecastConfig(max_context=336,max_horizon=48,normalize_inputs=True,per_core_batch_size=256))
    # Generation uses the unchanged model calls documented above. Implementations should
    # batch contexts, preserve 0.1/0.5/0.9 outputs, and write candidate CSV/NPZ files only to STAGING.


### 12.2 Candidate Validation

Any staged candidate must pass the same invariants as the frozen artifacts before it can be considered for a separately reviewed promotion.


In [ ]:
def validate_candidate_a(frame, model_column):
    return pd.Series({"Expected row count":len(frame)==len(test),"Timestamp alignment":frame.Timestamp.equals(pd.Series(test.index,name="Timestamp")),"Finite predictions":np.isfinite(frame[model_column]).all(),"No missing values":not frame[model_column].isna().any(),"Unique timestamps":frame.Timestamp.is_unique})
def validate_candidate_b(frame, model_column):
    return pd.Series({"Expected row count":len(frame)==len(test),"Timestamp alignment":frame.Timestamp.equals(pd.Series(test.index,name="Timestamp")),"Finite predictions":np.isfinite(frame[model_column]).all(),"No missing values":not frame[model_column].isna().any(),"962 origins":frame.Origin.nunique()==962,"48 values per origin":frame.groupby("Origin").size().eq(48).all(),"Horizons 1–48":frame.groupby("Origin").Horizon.apply(lambda z:z.tolist()==list(range(1,49))).all()})
frozen_checks=[]
for model,col in MODEL_COLUMNS.items():
    frozen_checks.append(validate_candidate_a(pa[["Timestamp",col]],col).rename((model,"A")))
    frozen_checks.append(validate_candidate_b(pb[["Origin","Timestamp","Horizon",col]],col).rename((model,"B")))
candidate_validation=pd.DataFrame(frozen_checks); display(candidate_validation)
assert candidate_validation.to_numpy().all()


### 12.3 Authoritative Promotion

`PROMOTE_TO_AUTHORITATIVE` remains `False`. Promotion is outside this notebook's normal path and requires explicit human review of row counts, origin/timestamp alignment, finite and non-missing predictions, actual-value identity, horizon conformity, and model-vector distinctness. This restructuring performs no promotion.


## 13. Foundation-Model Scope and Deferred Models

Unavailable or deferred families are recorded as research scope, not installation transcripts.


In [ ]:
registry=pd.DataFrame([
 {"Model":"Chronos-Bolt-Tiny","Category":"Pretrained foundation model","Evaluated?":"Yes","Reason":"Authoritative frozen forecasts available","Authoritative forecast?":"Yes","Future-work status":"Current evidence"},
 {"Model":"TimesFM","Category":"Pretrained foundation model","Evaluated?":"Yes","Reason":"Authoritative frozen forecasts available","Authoritative forecast?":"Yes","Future-work status":"Current evidence"},
 {"Model":"Moirai / Uni2TS","Category":"Deferred foundation-model family","Evaluated?":"No","Reason":"No authoritative Electricity forecast artifact in this repository","Authoritative forecast?":"No","Future-work status":"Potential extension after environment and protocol validation"}])
display(registry)


## 14. Key Findings

The statements below are generated from the computed tables rather than transcribed results.


In [ ]:
best_a=fm_a.iloc[0]; best_b=fm_b.iloc[0]
chronos_h_wins=int((hwin["Chronos-Bolt-Tiny"] < hwin["TimesFM"]).sum())
calibration_best=tradeoff.loc[tradeoff.groupby("Protocol")["Absolute coverage error"].idxmin()].set_index("Protocol")
rel=relative_improvement.set_index(["Protocol","Model"])
findings=[
 f"Within this South Australian Electricity case study, {best_a.Model} has the lower Protocol A foundation-model MASE-48 ({best_a['MASE-48']:.4f}).",
 f"Under genuine day-ahead Protocol B, {best_b.Model} has the lower foundation-model MASE-48 ({best_b['MASE-48']:.4f}).",
 f"Relative performance changes across the 48-step horizon: Chronos has lower MAE at {chronos_h_wins} horizons and TimesFM at {48-chronos_h_wins} horizons.",
 f"Against the strongest inspected Protocol A classical baseline ({best_a_baseline}), Chronos and TimesFM change MASE-48 by {rel.loc[('A','Chronos-Bolt-Tiny'),'Relative improvement %']:.1f}% and {rel.loc[('A','TimesFM'),'Relative improvement %']:.1f}%, respectively.",
 f"Against the strongest inspected Protocol B classical baseline ({best_b_baseline}), Chronos and TimesFM change MASE-48 by {rel.loc[('B','Chronos-Bolt-Tiny'),'Relative improvement %']:.1f}% and {rel.loc[('B','TimesFM'),'Relative improvement %']:.1f}%, respectively.",
 f"Point accuracy and native 80% calibration do not automatically select the same model: the smallest absolute coverage error belongs to {calibration_best.loc['A','Model']} in Protocol A and {calibration_best.loc['B','Model']} in Protocol B.",
 "Protocol A and Protocol B are not interchangeable: one permits half-hourly context updates, whereas the other fixes the information set for all 48 day-ahead predictions."
]
display(Markdown("\n".join(f"{i+1}. {text}" for i,text in enumerate(findings))))


## 15. Limitations

- Evidence covers one electricity region and only two authoritative foundation-model families, with one size/checkpoint per family.
- Evaluation is zero-shot only: there is no fine-tuning or systematic context-length sensitivity experiment.
- Unknown pretraining overlap or contamination cannot be excluded.
- Checkpoint revisions were not pinned or recorded, so environment and remote-checkpoint reproducibility remain risks even though repeated deterministic inference does not have the same multi-seed issue as neural training.
- No comprehensive runtime, memory, or energy benchmark is claimed; the prior notebook's transient measurements are not treated as frozen evidence.
- Native probabilistic evidence is limited to preserved 80% quantiles/aggregates; no 95% interval is available here.
- Protocol A and Protocol B answer distinct operational questions and must not be averaged or combined into one ranking.

These limitations bound the evidence; they do not invalidate the protocol-specific results.


## 16. Next Notebook

Next: `14_Electricity_Model_Validation_Audit.ipynb`

Notebook 13 establishes the foundation-model evidence. Notebook 14 independently checks forecast integrity, alignment, leakage controls, and the frozen comparison boundary before robustness, uncertainty, trustworthiness, and statistical inference are performed downstream.
